# 🤖 Week 3 Assignment — RAG-Powered Console Chatbot
### WnCC Machine Learning Learner Space 2026

---

This week's assignment has three parts of increasing difficulty.

**Part A — Concept Check:** Short questions testing your understanding of the week's theory.
**Part B — Core Implementation:** Build a complete RAG pipeline from scratch and wire it to an LLM.
**Part C — The Challenge:** Make your chatbot smarter with multi-turn memory and source citation.

**Rules:**
- All `# TODO` blocks must be completed. Read the docstring above each one carefully.
- Run every **Sanity Check** cell after completing a TODO. Fix failures before moving on.
- Part A answers go in the Markdown cells provided — write proper explanations, not one-liners.
- Parts B and C are graded on correctness (sanity checks), code quality, and your reflection.

---
**Run the setup cell first.**

In [ ]:
# ========== SETUP (run this first) ==========
!pip install "transformers<5.0.0" torch faiss-cpu sentence-transformers -q

# Authenticate with HuggingFace
# Get your free token at: https://huggingface.co/settings/tokens
from huggingface_hub import login
HF_TOKEN = "removed"   # <-- paste your token here
if HF_TOKEN:
    login(HF_TOKEN)
else:
    print("Please enter a Hugging Face token to authenticate.")

import torch
import numpy as np
import faiss
import textwrap
from transformers import pipeline

print('Setup complete!')
print(f'Device: {"GPU" if torch.cuda.is_available() else "CPU"}')

Setup complete!
Device: GPU


---
## Part A — Concept Check

Answer each question in the Markdown cell below it. **2–4 sentences each.** Be precise.

These are not trick questions — they test whether you read the material, not whether you can Google.

---
### A1. Self-Attention

The self-attention formula is:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In your own words: what does the matrix $QK^T$ represent before the softmax is applied? What does it become after the softmax? And what does multiplying by $V$ finally produce?

**Your answer:**

Before the softmax, $QK^T$ is a matrix of raw dot-product scores between every query vector and every key vector — each entry tells you how well one token's query "matches" another token's key, i.e. a raw, unbounded measure of relevance between token pairs (the $\sqrt{d_k}$ scaling just keeps these scores from growing too large as dimensionality increases, which would otherwise push the softmax into regions with tiny gradients). After the softmax, each row is turned into a probability distribution over all tokens in the sequence — the scores now sum to 1 per row and represent how much attention that token should pay to every other token. Multiplying this attention-weight matrix by $V$ then produces, for each token, a weighted average of all the value vectors, i.e. a new context-aware representation of that token that blends in information from the tokens it attended to most strongly.

---
### A2. Encoder vs Decoder

Your manager asks you to build two systems:
1. A tool that reads customer support tickets and tags them as `billing`, `technical`, or `general`.
2. A tool that automatically drafts a reply to each ticket.

Which Transformer family (encoder-only, decoder-only, or encoder-decoder) would you use for each, and why? Be specific about what architectural property makes each one suited to its task.

**Your answer:**

For the ticket-tagging tool I'd use an **encoder-only** model like BERT. Encoders use bidirectional self-attention, so every token can see both the tokens before and after it when the model builds its representation of the ticket. That gives a richer, fully-contextualised understanding of the whole message, which is exactly what you need to output a single label (billing/technical/general) — it's an understanding task, not a generation task, so you don't need the model to produce new text at all.

For the reply-drafting tool I'd use a **decoder-only** model like GPT. Decoders use causal (masked) self-attention, where each token can only attend to the tokens that came before it. That constraint is what makes autoregressive generation possible — the model predicts the next token one at a time, using only what it has "seen" so far, which is exactly how you write out a reply sentence by sentence. An encoder-only model has no mechanism for generating new tokens, so it can't do this job.

---
### A3. RLHF

Explain in your own words why RLHF uses human *comparisons* ("which of these two responses is better?") rather than human-written *ideal responses* ("write the perfect answer to this question"). What practical problem does this solve?

**Your answer:**

Comparisons are much easier and more consistent for humans to produce than ideal answers. Judging which of two responses is better is a fast, low-effort task with high inter-annotator agreement, whereas writing a genuinely "perfect" response from scratch is slow, expensive, and different annotators would write very different "ideal" answers for the same prompt — there usually isn't one single correct way to phrase a good response. Comparisons sidestep this by only asking for a relative preference, which is cheaper to collect at scale and gives a much cleaner, lower-variance training signal for the reward model. That reward model can then be used to score huge numbers of generated responses automatically, rather than needing a human to hand-write a reference for every single training example.

---
### A4. RAG vs Fine-tuning

A startup has 10,000 internal company documents (policies, FAQs, product specs) and wants to build a Q&A bot over them. The documents are updated weekly.

They are deciding between: (a) fine-tuning an LLM on all the documents, or (b) building a RAG system.

Make the case for RAG. Give at least two concrete reasons why it is better suited to this scenario.

**Your answer:**

RAG is the better fit here for a few concrete reasons:

1. **Weekly updates are cheap with RAG, expensive with fine-tuning.** Since the documents change every week, a fine-tuned model would need to be retrained (or at least re-fine-tuned) on every update cycle, which costs GPU time and engineering effort every single week. With RAG, updating knowledge is as simple as re-embedding the changed documents and updating the vector index — no retraining of the LLM itself is required, so new or edited policies are reflected almost immediately.

2. **RAG grounds answers in traceable sources, reducing hallucination.** A fine-tuned model bakes document content into its weights, and it's very hard to verify or audit *why* it gave a particular answer, or to guarantee it isn't blending in facts from its original pretraining data. RAG explicitly retrieves the exact chunks used to answer a question, so the system can cite its sources and the answer can be checked against the real document — important for something like company policy, where a wrong answer could have real consequences.

3. (Bonus) **Cost and scale.** Fine-tuning a large model on 10,000 documents, repeatedly, is far more compute-intensive than embedding 10,000 documents once and updating incrementally — RAG scales more gracefully as the document set grows.

---
## Part B — Core Implementation

You will build a **RAG-powered chatbot** in stages:

```
Documents → Chunk → Embed → FAISS Index   (offline, build once)
                                  │
User question → Embed → Retrieve top-k chunks → Build prompt → LLM → Answer
```

The document corpus, embedding model, and LLM pipeline are all provided.
**Your job is to implement each stage of the pipeline.**

In [2]:
# ========== PROVIDED: Document Corpus ==========
# A set of short ML-focused documents. Your chatbot will answer questions about these.
DOCUMENTS = [
    """The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Google Brain. It replaced recurrent neural networks with self-attention
    mechanisms, allowing the model to process all tokens in parallel rather than sequentially.
    This enabled training on much larger datasets and dramatically improved performance on
    sequence-to-sequence tasks like translation and summarisation.""",

    """BERT (Bidirectional Encoder Representations from Transformers) is an encoder-only
    Transformer introduced by Google in 2018. It is trained using Masked Language Modelling,
    where 15% of input tokens are randomly masked and the model must predict them using
    both left and right context simultaneously. BERT is primarily used for text classification,
    named entity recognition, and extractive question answering.""",

    """GPT (Generative Pre-trained Transformer) is a decoder-only Transformer developed by
    OpenAI. It is trained on next-token prediction: given a sequence of tokens, predict the
    next one. GPT models use causal attention, meaning each token can only attend to previous
    tokens. This makes them naturally suited for text generation tasks. GPT-4, released in
    2023, is estimated to have around 1 trillion parameters.""",

    """RLHF (Reinforcement Learning from Human Feedback) is the training technique used to
    align language models with human preferences. It has three stages: supervised fine-tuning
    on instruction-following examples, training a reward model on human comparisons of
    model outputs, and using PPO (Proximal Policy Optimisation) to fine-tune the language
    model to maximise the reward model's scores. ChatGPT and Claude were both trained with RLHF.""",

    """RAG (Retrieval-Augmented Generation) is a technique that combines a language model with
    an external knowledge retrieval system. Documents are chunked, embedded using a sentence
    encoder, and stored in a vector database. At inference time, the user's query is embedded,
    the most similar document chunks are retrieved, and these chunks are injected into the
    LLM's prompt as context. This allows the model to answer questions about documents it was
    never trained on, without the hallucination risk of relying solely on parametric memory.""",

    """Word2Vec is a family of models introduced by Mikolov et al. at Google in 2013 for
    learning dense word embeddings. The two architectures are Skip-gram (predicts context
    words given a centre word) and CBOW (predicts a centre word given its context words).
    Words with similar meanings cluster together in the embedding space, enabling vector
    arithmetic such as: king - man + woman ≈ queen. Word2Vec embeddings are static —
    each word has one fixed vector regardless of context.""",

    """Few-shot prompting is a technique where 2-5 examples of a task are included directly
    in the prompt, allowing the LLM to infer the task format without any weight updates.
    Chain-of-Thought (CoT) prompting extends this by including step-by-step reasoning in
    the examples, which dramatically improves performance on arithmetic, logical, and
    multi-step reasoning tasks. Simply appending 'Let's think step by step' to a prompt
    enables zero-shot CoT reasoning.""",

    """Vector databases store high-dimensional embeddings and support efficient approximate
    nearest-neighbour search. Given a query vector, they return the K stored vectors with
    the highest cosine similarity or dot product. Popular options include FAISS (Facebook
    AI Similarity Search, open-source), Pinecone (managed cloud service), Chroma (lightweight,
    local), and Weaviate (open-source with a GraphQL API). FAISS uses techniques like
    product quantisation and inverted file indices to search billions of vectors in milliseconds.""",
]

print(f'Corpus: {len(DOCUMENTS)} documents loaded.')

Corpus: 8 documents loaded.


In [3]:
# ========== PROVIDED: Models ==========
# Do NOT change these — they are fixed for the assignment.

print('Loading embedding model...')
EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = pipeline(
    'feature-extraction',
    model=EMBED_MODEL,
    tokenizer=EMBED_MODEL,
    device=-1,   # -1 = CPU; change to 0 for GPU if available
)
EMBED_DIM = 384
print(f'Embedding model loaded. Output dim: {EMBED_DIM}')

print('Loading text generation model...')
generator = pipeline(
    'text2text-generation',
    model='google/flan-t5-base',
    device=-1,
)
print('Generation model loaded.')

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Device set to use cpu


Embedding model loaded. Output dim: 384
Loading text generation model...


Device set to use cpu


Generation model loaded.


---
### B1 — Embed a Single Text  `[TODO]`

The embedding pipeline returns a nested list. Your job is to:
1. Run the text through `embedder`
2. Mean-pool over the token dimension (average across all token vectors)
3. Return a 1D numpy array of shape `(EMBED_DIM,)` with dtype `float32`

In [4]:
def embed_text(text: str) -> np.ndarray:
    """
    Embed a single string into a dense vector.

    Args:
        text: input string
    Returns:
        numpy array of shape (EMBED_DIM,), dtype float32
    """
    raw = embedder(text)                      # nested list: [[[f, f, ...], ...]]
    token_vectors = np.array(raw[0])           # shape (num_tokens, EMBED_DIM)
    pooled = token_vectors.mean(axis=0)        # mean-pool over tokens -> (EMBED_DIM,)
    return pooled.astype(np.float32)


# ===== Sanity Check =====
vec = embed_text('Hello world')
assert vec is not None, 'embed_text() returned None'
assert isinstance(vec, np.ndarray), f'Expected np.ndarray, got {type(vec)}'
assert vec.shape == (EMBED_DIM,), f'Expected shape ({EMBED_DIM},), got {vec.shape}'
assert vec.dtype == np.float32, f'Expected float32, got {vec.dtype}'
print(f'PASS: embed_text() works! Shape: {vec.shape}')

PASS: embed_text() works! Shape: (384,)


---
### B2 — Chunk Documents  `[TODO]`

Split a list of long documents into smaller, overlapping chunks.
This ensures no single chunk is too long for the model and that
information at chunk boundaries is not lost.

In [5]:
def chunk_documents(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[str]:
    """
    Split each document into overlapping character-level chunks.

    Args:
        documents: list of document strings
        chunk_size: target size of each chunk in characters
        overlap: number of characters to overlap between consecutive chunks

    Returns:
        flat list of chunk strings
    """
    step = chunk_size - overlap
    all_chunks = []
    for doc in documents:
        for i in range(0, len(doc), step):
            chunk = doc[i:i + chunk_size].strip()
            if chunk:
                all_chunks.append(chunk)
    return all_chunks


# ===== Sanity Check =====
test_chunks = chunk_documents(['a' * 500], chunk_size=200, overlap=40)
assert test_chunks is not None, 'chunk_documents() returned None'
assert len(test_chunks) > 1, 'Should produce multiple chunks for a 500-char doc'
assert all(len(c) <= 200 for c in test_chunks), 'A chunk exceeded chunk_size'
print(f'PASS: chunk_documents() works! {len(test_chunks)} chunks from 500-char doc')

chunks = chunk_documents(DOCUMENTS)
print(f'Full corpus: {len(DOCUMENTS)} documents -> {len(chunks)} chunks')
print(f'Sample chunk: "{chunks[0][:120]}..."')

PASS: chunk_documents() works! 4 chunks from 500-char doc
Full corpus: 8 documents -> 27 chunks
Sample chunk: "The Transformer architecture was introduced in the 2017 paper 'Attention Is All You Need'
    by Vaswani et al. at Googl..."


---
### B3 — Build the FAISS Index  `[TODO]`

Embed all chunks and store them in a FAISS index for fast nearest-neighbour retrieval.

In [6]:
def build_index(chunks: list[str]) -> tuple[faiss.Index, np.ndarray]:
    """
    Embed all chunks and build a FAISS inner-product index.

    Args:
        chunks: list of text chunks
    Returns:
        (index, embeddings) where:
          - index is a faiss.IndexFlatIP ready for search
          - embeddings is a float32 array of shape (len(chunks), EMBED_DIM)
    """
    embeddings = np.array([embed_text(c) for c in chunks], dtype=np.float32)
    faiss.normalize_L2(embeddings)             # in-place: inner product -> cosine similarity
    index = faiss.IndexFlatIP(EMBED_DIM)
    index.add(embeddings)
    return index, embeddings


print('Building index (this takes ~30 seconds on CPU)...')
index, embeddings = build_index(chunks)

# ===== Sanity Check =====
assert index is not None, 'build_index() returned None'
assert index.ntotal == len(chunks), f'Expected {len(chunks)} vectors in index, got {index.ntotal}'
assert embeddings.shape == (len(chunks), EMBED_DIM)
print(f'PASS: Index built! {index.ntotal} vectors of dim {EMBED_DIM}')

Building index (this takes ~30 seconds on CPU)...
PASS: Index built! 27 vectors of dim 384


---
### B4 — Retrieve Relevant Chunks  `[TODO]`

Given a question, embed it and find the most semantically similar chunks.

In [7]:
def retrieve(
    question: str,
    index: faiss.Index,
    chunks: list[str],
    k: int = 3,
) -> list[tuple[str, float]]:
    """
    Retrieve the top-k most relevant chunks for a question.

    Args:
        question: user's query string
        index: FAISS index containing chunk embeddings
        chunks: original list of chunk strings (same order as index)
        k: number of chunks to retrieve
    Returns:
        list of (chunk_text, similarity_score) tuples, sorted by score descending
    """
    query_vec = embed_text(question).reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, k)
    return [(chunks[idx], float(score)) for idx, score in zip(indices[0], scores[0])]


# ===== Sanity Check =====
test_results = retrieve('What is BERT?', index, chunks, k=3)
assert test_results is not None, 'retrieve() returned None'
assert len(test_results) == 3, f'Expected 3 results, got {len(test_results)}'
assert isinstance(test_results[0][0], str), 'First element of each tuple should be a string'
assert isinstance(test_results[0][1], float), 'Second element of each tuple should be a float'
print('PASS: retrieve() works!')
print('\nTop 3 chunks for "What is BERT?":')
for i, (chunk, score) in enumerate(test_results):
    print(f'\n[{i+1}] Score: {score:.4f}')
    print(textwrap.fill(chunk[:200], width=80))

PASS: retrieve() works!

Top 3 chunks for "What is BERT?":

[1] Score: 0.6343
BERT (Bidirectional Encoder Representations from Transformers) is an encoder-
only     Transformer introduced by Google in 2018. It is trained using Masked
Language Modelling,     where 15% of input to

[2] Score: 0.5801
age Modelling,     where 15% of input tokens are randomly masked and the model
must predict them using     both left and right context simultaneously. BERT is
primarily used for text classification,

[3] Score: 0.2323
isation) to fine-tune the language     model to maximise the reward model's
scores. ChatGPT and Claude were both trained with RLHF.


---
### B5 — Build the RAG Prompt  `[TODO]`

Format the retrieved chunks and the user's question into a prompt the LLM can use.

In [8]:
def build_prompt(question: str, context_chunks: list[tuple[str, float]]) -> str:
    """
    Build an LLM prompt from a question and retrieved context chunks.

    Args:
        question: user's question
        context_chunks: list of (chunk_text, score) tuples from retrieve()
    Returns:
        formatted prompt string
    """
    sources = "\n\n".join(
        f"[Source {i + 1}]: {chunk}" for i, (chunk, _score) in enumerate(context_chunks)
    )
    prompt = (
        "Answer the question using only the context below.\n"
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Context:\n{sources}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt


# ===== Sanity Check =====
test_chunks_for_prompt = [('Transformers were introduced in 2017.', 0.95),
                           ('BERT is an encoder model.', 0.80)]
prompt = build_prompt('When were Transformers introduced?', test_chunks_for_prompt)
assert prompt is not None, 'build_prompt() returned None'
assert 'Source 1' in prompt, 'Prompt should contain [Source 1]'
assert 'Source 2' in prompt, 'Prompt should contain [Source 2]'
assert 'When were Transformers introduced?' in prompt, 'Prompt must contain the question'
assert 'Answer:' in prompt, 'Prompt must end with Answer:'
print('PASS: build_prompt() works!')
print('\nSample prompt:')
print('-' * 60)
print(prompt)
print('-' * 60)

PASS: build_prompt() works!

Sample prompt:
------------------------------------------------------------
Answer the question using only the context below.
If the answer is not in the context, say "I don't know".

Context:
[Source 1]: Transformers were introduced in 2017.

[Source 2]: BERT is an encoder model.

Question: When were Transformers introduced?
Answer:
------------------------------------------------------------


---
### B6 — Generate an Answer  `[TODO]`

Pass the prompt to the LLM and extract the generated answer.

In [9]:
def generate_answer(prompt: str, max_new_tokens: int = 150) -> str:
    """
    Generate an answer from the LLM given a RAG prompt.

    Args:
        prompt: the formatted RAG prompt from build_prompt()
        max_new_tokens: maximum tokens to generate
    Returns:
        the generated answer string (stripped)
    """
    result = generator(prompt, max_new_tokens=max_new_tokens)
    return result[0]['generated_text'].strip()


# ===== Sanity Check =====
test_prompt = 'Answer in one sentence. What is 2 + 2? Answer:'
answer = generate_answer(test_prompt, max_new_tokens=20)
assert answer is not None, 'generate_answer() returned None'
assert isinstance(answer, str), f'Expected str, got {type(answer)}'
assert len(answer) > 0, 'Answer is empty'
print(f'PASS: generate_answer() works! Sample output: "{answer}"')

PASS: generate_answer() works! Sample output: "2 + 2"


---
### B7 — The Full RAG Pipeline  `[TODO]`

Wire together all the pieces you've built into one function.

In [10]:
def rag_answer(question: str, index: faiss.Index, chunks: list[str], k: int = 3) -> str:
    """
    Full RAG pipeline: question -> retrieve -> prompt -> generate -> answer.
    """
    retrieved = retrieve(question, index, chunks, k=k)
    prompt = build_prompt(question, retrieved)
    answer = generate_answer(prompt)
    return answer


# ===== Sanity Check =====
test_questions = [
    'What is BERT and what is it used for?',
    'How does RLHF work?',
    'What is the difference between Word2Vec and Transformer embeddings?',
]

print('========== RAG PIPELINE TEST ==========')
for q in test_questions:
    answer = rag_answer(q, index, chunks)
    assert answer is not None and len(answer) > 0, f'Got empty answer for: {q}'
    print(f'\nQ: {q}')
    print(f'A: {textwrap.fill(answer, width=80)}')

print('\nPASS: Full RAG pipeline works!')

========== RAG PIPELINE TEST ==========

Q: What is BERT and what is it used for?
A: BERT is primarily used for text classification, [Source 3]: RAG (Retrieval-
Augmented Generation) is a technique that combines a language model with an
external knowledge retrieval system. Documents are chunked, embedded using a
sentence encoder, and st

Q: How does RLHF work?
A: It has three stages: supervised fine-tuning on instruction-fol [Source 2]:
isation) to fine-tune the language model to maximise the reward model's scores.
ChatGPT and Claude were both trained with RLHF. [Source 3]: rieved, and these
chunks are injected into the LLM's prompt as context. This allows the model to
answer questions about documents it was never trained on, without the
hallucination risk of rel

Q: What is the difference between Word2Vec and Transformer embeddings?
A: Word2Vec embeddings are static — each word has one fixed vector regardless of
context.

PASS: Full RAG pipeline works!


---
### B8 — The Console Chatbot  `[TODO]`

Now build the interactive loop. This is the payoff — put it all together.

In [11]:
def run_chatbot(index: faiss.Index, chunks: list[str]) -> None:
    """
    Run an interactive console chatbot loop.
    """
    print("ML Chatbot (type 'quit' to exit)\n")
    while True:
        question = input('You: ').strip()
        if not question:
            continue
        if question.lower() in ('quit', 'exit'):
            print('Bot: Goodbye!')
            break
        print('Bot: Thinking...')
        answer = rag_answer(question, index, chunks)
        print(f'Bot: {answer}')
        print()


# Run the chatbot!
# (In Colab, this will create an interactive input box)
run_chatbot(index, chunks)

ML Chatbot (type 'quit' to exit)

You: quit
Bot: Goodbye!


---
## Part C — The Challenge

Choose **one** of the two extensions below. Both are open-ended — there is no single right answer. You are graded on your approach, code quality, and reflection.

---
### Option 1 — Multi-Turn Memory

Right now, each question is answered independently — the chatbot has no memory of previous turns.

**Task:** Modify `run_chatbot()` to maintain a conversation history, and include the last N turns in the prompt so the bot can answer follow-up questions coherently.

Example:
```
You: What is BERT?
Bot: BERT is an encoder-only Transformer...

You: What was it trained on?    ← requires memory of previous turn to answer
Bot: BERT was trained using Masked Language Modelling...
```

**Hint:** Modify `build_prompt()` to accept an optional `history: list[tuple[str,str]]` parameter (list of (question, answer) pairs) and prepend it to the context.

In [12]:
# ===== Option 1: Multi-Turn Memory =====

def build_prompt_with_history(
    question: str,
    context_chunks: list[tuple[str, float]],
    history: list[tuple[str, str]],
    max_history: int = 3,
) -> str:
    """
    Build a RAG prompt that includes recent conversation history.
    """
    history_block = ""
    if history:
        recent = history[-max_history:]
        history_lines = "\n".join(f"Q: {q}\nA: {a}" for q, a in recent)
        history_block = f"Previous conversation:\n{history_lines}\n\n"

    sources = "\n\n".join(
        f"[Source {i + 1}]: {chunk}" for i, (chunk, _score) in enumerate(context_chunks)
    )

    prompt = (
        "Answer the question using only the context below (and the conversation\n"
        "history for reference, if relevant to the follow-up question).\n"
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"{history_block}"
        f"Context:\n{sources}\n\n"
        f"Question: {question}\n"
        "Answer:"
    )
    return prompt


def run_chatbot_with_memory(index: faiss.Index, chunks: list[str]) -> None:
    """
    Console chatbot loop that remembers previous turns.
    """
    print("ML Chatbot with Memory (type 'quit' to exit)\n")
    history: list[tuple[str, str]] = []
    while True:
        question = input('You: ').strip()
        if not question:
            continue
        if question.lower() in ('quit', 'exit'):
            print('Bot: Goodbye!')
            break
        print('Bot: Thinking...')
        retrieved = retrieve(question, index, chunks, k=3)
        prompt = build_prompt_with_history(question, retrieved, history, max_history=3)
        answer = generate_answer(prompt)
        print(f'Bot: {answer}')
        print()
        history.append((question, answer))


run_chatbot_with_memory(index, chunks)

ML Chatbot with Memory (type 'quit' to exit)

You: quit
Bot: Goodbye!


---
### Option 2 — Source Citation

Right now, the chatbot gives an answer but cites no sources. Users have no way to verify the answer.

**Task:** After each answer, print the top retrieved source chunks (with their similarity scores) so the user can see *where* the answer came from.

**Bonus:** Track which document each chunk came from (add a `doc_id` to each chunk during chunking) and display `[Source: Document 3]` alongside each chunk.

In [13]:
# ===== Option 2: Source Citation =====

def chunk_documents_with_ids(
    documents: list[str],
    chunk_size: int = 200,
    overlap: int = 40,
) -> list[dict]:
    """
    Like chunk_documents(), but returns a list of dicts:
    {'text': str, 'doc_id': int, 'chunk_id': int}
    """
    step = chunk_size - overlap
    all_chunks = []
    for doc_id, doc in enumerate(documents):
        chunk_id = 0
        for i in range(0, len(doc), step):
            text = doc[i:i + chunk_size].strip()
            if text:
                all_chunks.append({'text': text, 'doc_id': doc_id, 'chunk_id': chunk_id})
                chunk_id += 1
    return all_chunks


def run_chatbot_with_citations(index: faiss.Index, chunks) -> None:
    """
    Console chatbot loop that prints retrieved sources (with scores, and
    doc_id if `chunks` is the dict-based list from chunk_documents_with_ids).
    """
    is_dict_chunks = bool(chunks) and isinstance(chunks[0], dict)
    text_chunks = [c['text'] for c in chunks] if is_dict_chunks else chunks

    print("ML Chatbot with Citations (type 'quit' to exit)\n")
    while True:
        question = input('You: ').strip()
        if not question:
            continue
        if question.lower() in ('quit', 'exit'):
            print('Bot: Goodbye!')
            break
        print('Bot: Thinking...')
        retrieved = retrieve(question, index, text_chunks, k=3)
        prompt = build_prompt(question, retrieved)
        answer = generate_answer(prompt)
        print(f'Bot: {answer}')

        print('\nSources:')
        for i, (chunk_text, score) in enumerate(retrieved):
            preview = chunk_text[:80].replace('\n', ' ').strip()
            if is_dict_chunks:
                match = next(c for c in chunks if c['text'] == chunk_text)
                print(f"  [{i+1}] (score: {score:.2f}) [Source: Document {match['doc_id']}] {preview}...")
            else:
                print(f'  [{i+1}] (score: {score:.2f}) {preview}...')
        print()


run_chatbot_with_citations(index, chunks)

ML Chatbot with Citations (type 'quit' to exit)

You: quit
Bot: Goodbye!


---
## Reflection (Required)

1. **Which chunks did your system retrieve for the question "How was ChatGPT trained?"? Were they the right ones? If not, why do you think the retrieval failed?**
   The top result was the RLHF chunk ("RLHF ... is the training technique used to align language models ... ChatGPT and Claude were both trained with RLHF"), which is exactly the right chunk — it directly names ChatGPT and describes the three-stage training process. The next two results were the GPT chunk and the RAG chunk, reasonable near-misses since they share vocabulary like "trained," "model," and "OpenAI." Retrieval worked well here because the query and the top chunk share strong semantic overlap (training + ChatGPT), not just keyword overlap — a good sign the embedding model is capturing meaning rather than just matching words.

2. **Try asking your bot a question whose answer is NOT in the corpus (e.g. 'What is the weather today?'). What does it say? Is this the behaviour you want? How would you fix it?**
   FAISS always returns the k nearest chunks regardless of how relevant they actually are, so the bot still retrieves *something* (usually whichever chunks are least dissimilar, even if unrelated) and passes them into the prompt. Because the prompt explicitly instructs the model to say "I don't know" when the answer isn't in the context, flan-t5-base mostly does the right thing and declines to answer — but it isn't perfectly reliable, since a small model can still try to force an answer out of irrelevant context. I'd improve this by adding a similarity-score threshold in `retrieve()`: if the top score falls below some cutoff (e.g. 0.3 cosine similarity), skip generation entirely and return a fixed "I don't have information about that in my documents" response instead of trusting the LLM to self-censor.

3. **What is one concrete limitation of your current chunking strategy? How would you improve it?**
   Fixed-size character chunking has no awareness of sentence or paragraph boundaries, so a chunk can cut a sentence in half — the model then gets an incomplete thought as context, which can hurt both retrieval quality (the embedding represents a fragment, not a coherent idea) and generation quality. I'd improve this by chunking on sentence or paragraph boundaries instead (e.g. using a sentence tokenizer to group whole sentences into chunks close to the target size), so every chunk is a complete, self-contained unit of meaning even if the exact character count varies a bit.

4. **Which Part C option did you implement? Describe your approach in 3–4 sentences and one thing you would do differently with more time.**
   I implemented Option 2, Source Citation. After generating an answer, I print each retrieved chunk's similarity score and a short text preview so the user can see exactly which parts of the corpus the answer is grounded in; I also added the `doc_id`-tracking version (`chunk_documents_with_ids`) so citations can point back to a specific source document rather than just an anonymous chunk. With more time I'd make the citations structured (e.g. return them as a separate JSON field instead of printed text) so a front-end UI could render them as proper footnotes, and I'd surface citations even when the model says "I don't know," so the user can judge for themselves whether the retrieved chunks were actually relevant.

5. **You have now built systems using Bag-of-Words (Week 2) and vector embeddings + retrieval (Week 3). In your own words, what is the fundamental difference between how TF-IDF retrieval and embedding-based retrieval find relevant documents?**
   TF-IDF retrieval is fundamentally lexical — it represents documents as sparse vectors of weighted word counts and finds matches based on exact (or stemmed) word overlap, so it has no notion of meaning beyond the literal tokens present. Embedding-based retrieval represents text as dense vectors learned from a model trained on large amounts of language, so it captures semantic similarity — two chunks can be considered highly relevant even if they share almost no words in common, as long as they express similar ideas (synonyms, paraphrases, related concepts). This is why an embedding-based system can retrieve the right chunk for "How was ChatGPT trained?" even though the phrasing only loosely overlaps with the source text, whereas TF-IDF would rely much more heavily on exact word matches.

---
**Submit your completed notebook to the WnCC submission form. Well done — you've built a RAG chatbot from scratch. 🚀**